# J-lens fit — Qwen 3.5-4B (scheduled Kaggle run)

Runs end-to-end without human intervention. Uploaded via `kaggle kernels push`. Outputs:
- `/kaggle/working/qwen-3.5-4b-jlens.pt` — the fitted Jacobian lens
- `/kaggle/working/eval_results.json` — pass@k on the 6 lens-quality evaluations
- `/kaggle/working/probe_readouts.json` — 3 sanity-check readouts (spider/Italy/Canada)

Pull results with: `kaggle kernels output amaljithkuttamath/jlens-qwen-3-6-4b -p out/`

In [ ]:
import subprocess, sys, torch
# Kaggle assigns either P100 (sm_60) or T4 (sm_75). Kaggle's preinstalled
# PyTorch is built for sm_70+ only, so on P100 we get 'no kernel image' errors.
# Reinstall a torch build that includes sm_60. cu118 wheels cover both P100 and T4.
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print('GPU compute capability:', cap)
if cap == (6, 0):
    print('P100 detected — reinstalling PyTorch with cu118 (covers sm_60).')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
        'torch==2.4.1'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/anthropics/jacobian-lens.git',
    'datasets', 'safetensors', '--upgrade', 'transformers>=5.5'])


In [ ]:
import os, json, time, pathlib
import torch, transformers, jlens
from datasets import load_dataset

MODEL = 'Qwen/Qwen3.5-4B'
N_PROMPTS = 25
MAX_SEQ_LEN = 128
SKIP_FIRST = 4
TARGET_LAYER = -2
DIM_BATCH = 4
OUT_DIR = pathlib.Path('/kaggle/working')
OUT_LENS = OUT_DIR / 'qwen-3.5-4b-jlens.pt'

print('CUDA:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}]', torch.cuda.get_device_name(i))

In [ ]:
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa',
    low_cpu_mem_usage=True,
).to('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
model = jlens.from_hf(hf_model, tokenizer, compile=True)
print(f'n_layers={model.n_layers}  d_model={model.d_model}')

In [ ]:
ds = load_dataset('NeelNanda/pile-10k', split='train')
prompts = []
for row in ds:
    if isinstance(row['text'], str) and len(row['text']) > 200:
        prompts.append(row['text'])
        if len(prompts) >= N_PROMPTS: break
print(f'Selected {len(prompts)} prompts')

In [ ]:
t0 = time.time()
lens = jlens.fit(
    model, prompts=prompts,
    target_layer=TARGET_LAYER, dim_batch=DIM_BATCH,
    max_seq_len=MAX_SEQ_LEN, skip_first=SKIP_FIRST,
    checkpoint_path=str(OUT_DIR / 'ckpt.pt'),
)
print(f'Fit done in {(time.time()-t0)/60:.1f} min')
print(lens)
lens.save(str(OUT_LENS))
print('Peak VRAM:', torch.cuda.max_memory_allocated()/1e9, 'GB')

In [ ]:
# --- Sanity check readouts ---
probes = [
    'The number of legs on the animal that spins webs is',
    'The capital of the country shaped like a boot is',
    'Fact: The currency used in the country whose flag has a red maple leaf is the',
]
probe_out = {}
for prompt in probes:
    lens_logits, _, _ = lens.apply(model, prompt, positions=[-2])
    per_layer = {}
    for layer in sorted(lens_logits):
        top = lens_logits[layer][0].topk(5).indices.tolist()
        per_layer[layer] = [tokenizer.decode([t]).strip() for t in top]
    probe_out[prompt] = per_layer
with open(OUT_DIR / 'probe_readouts.json', 'w') as f:
    json.dump(probe_out, f, indent=2)
print('Wrote probe_readouts.json')

In [ ]:
# --- Lens-quality evals: multihop, multilingual, poetry, order-of-ops, association, typo ---
!git clone -q --depth=1 https://github.com/anthropics/jacobian-lens.git /tmp/jl
EVALS_DIR = pathlib.Path('/tmp/jl/data/evaluations')

def tokens_of(word):
    ids = []
    for v in {word, ' '+word, word.lower(), ' '+word.lower()}:
        e = tokenizer(v, add_special_tokens=False).input_ids
        if len(e) == 1: ids.append(e[0])
    return sorted(set(ids))

def min_rank_across_layers(lens_logits, token_ids):
    best = 10**9
    for _, logits in lens_logits.items():
        order = logits[0].argsort(descending=True)
        rank_lookup = torch.empty_like(order)
        rank_lookup[order] = torch.arange(order.numel(), device=order.device)
        cand = rank_lookup[torch.tensor(token_ids, device=order.device)]
        r = int(cand.min().item()) + 1
        if r < best: best = r
    return best

K_VALUES = (1, 5, 10)
eval_results = []
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    data = json.load(path.open())
    items = data['items']
    per_item_frac = {k: [] for k in K_VALUES}
    for item in items:
        try:
            lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
        except Exception as e:
            continue
        inters = item['intermediates'] if isinstance(item['intermediates'], list) else [item['intermediates']]
        hits = {k: 0 for k in K_VALUES}
        total = 0
        for inter in inters:
            key = inter if isinstance(inter, str) else (inter.get('synonyms', [inter.get('key','')])[0] if isinstance(inter, dict) else inter[0])
            tok_ids = tokens_of(str(key))
            if not tok_ids: continue
            r = min_rank_across_layers(lens_logits, tok_ids)
            for k in K_VALUES:
                if r <= k: hits[k] += 1
            total += 1
        if total:
            for k in K_VALUES: per_item_frac[k].append(hits[k]/total)
    result = {
        'eval': path.stem,
        'n_items_scored': len(per_item_frac[1]),
        'pass_at_k': {f'pass@{k}': (sum(per_item_frac[k])/max(1,len(per_item_frac[k]))) for k in K_VALUES},
    }
    eval_results.append(result)
    print(path.name, '→', result['pass_at_k'])

with open(OUT_DIR / 'eval_results.json', 'w') as f:
    json.dump({'model': MODEL, 'n_prompts': N_PROMPTS, 'results': eval_results}, f, indent=2)
print('\nDone. Outputs in /kaggle/working/:')
for p in OUT_DIR.iterdir():
    print(' ', p.name, f'({p.stat().st_size/1e6:.1f} MB)')